# 01_03 - GNOME Data Collection

## Mục tiêu

Notebook này thực hiện bước **Data Collection** cho source:

**GNOME — OPUS**

Các nhiệm vụ:

1. Kiểm tra môi trường
2. Xác định project root
3. Cấu hình source
4. Download raw corpus
5. Giải nén corpus
6. Kiểm tra cấu trúc dữ liệu
7. Convert parallel data sang DataFrame
8. Inspect raw data
9. Thống kê raw data
10. Xác định technology candidate
11. Tổng hợp audit summary
12. Lưu raw JSONL
13. Lưu raw Parquet
14. Lưu audit summary
15. Lưu metadata
16. Final verification
17. Ghi trạng thái notebook

> Lưu ý:
> - Đây là corpus parallel từ OPUS
> - Không cleaning
> - Không deduplication
> - Không train/validation/test split
> - Không overwrite raw source
> - Technology candidate chưa đồng nghĩa với usable IT corpus
> - `usable_count` chưa được xác định


In [1]:
import sys
import os
from pathlib import Path

print("Python:", sys.version)
print("Working directory:", os.getcwd())

Python: 3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]
Working directory: C:\Users\ADMIN\ENVI-IT-MT\notebooks\01_data_collection


In [2]:
import datasets
import pandas as pd
import httpx

print("datasets:", datasets.__version__)
print("pandas:", pd.__version__)
print("httpx:", httpx.__version__)

datasets: 5.0.1
pandas: 3.0.5
httpx: 0.28.1


In [3]:
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()


def find_project_root(start_path: Path) -> Path:
    """
    Tìm project root bằng cách kiểm tra:
    - data/
    - notebooks/ hoặc notebook/
    """
    candidates = [start_path] + list(start_path.parents)

    for path in candidates:
        if (
            (path / "data").is_dir()
            and (
                (path / "notebooks").is_dir()
                or (path / "notebook").is_dir()
            )
        ):
            return path

    raise FileNotFoundError(
        "Không tìm thấy project root. "
        "Hãy kiểm tra lại vị trí notebook."
    )


PROJECT_ROOT = find_project_root(CURRENT_DIR)

print("Project root:")
print(PROJECT_ROOT)

Project root:
C:\Users\ADMIN\ENVI-IT-MT


In [4]:
SOURCE_NAME = "GNOME"

SOURCE_SHORT_NAME = "gnome"

SOURCE_URL = (
    "https://opus.nlpl.eu/GNOME/corpus/version/GNOME"
)

DOWNLOAD_URL = (
    "https://object.pouta.csc.fi/"
    "OPUS-GNOME/v1/moses/en-vi.txt.zip"
)

LANGUAGE_PAIR = "en-vi"

DOMAIN = "IT / Software Localization"

DATASET_VERSION = "v1"

DOWNLOAD_METHOD = (
    "OPUS direct download - Moses format"
)

COLLECTION_DATE = pd.Timestamp.now().strftime("%Y-%m-%d")

RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / SOURCE_SHORT_NAME
)

RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Source:", SOURCE_NAME)
print("Source URL:", SOURCE_URL)
print("Download URL:", DOWNLOAD_URL)
print("Language pair:", LANGUAGE_PAIR)
print("Domain:", DOMAIN)
print("Dataset version:", DATASET_VERSION)
print("Raw directory:", RAW_DIR)

Source: GNOME
Source URL: https://opus.nlpl.eu/GNOME/corpus/version/GNOME
Download URL: https://object.pouta.csc.fi/OPUS-GNOME/v1/moses/en-vi.txt.zip
Language pair: en-vi
Domain: IT / Software Localization
Dataset version: v1
Raw directory: C:\Users\ADMIN\ENVI-IT-MT\data\raw\gnome


In [5]:
import zipfile

zip_path = RAW_DIR / "en-vi.txt.zip"

response = httpx.get(
    DOWNLOAD_URL,
    follow_redirects=True,
    timeout=120
)

response.raise_for_status()

zip_path.write_bytes(response.content)

print("Downloaded:")
print(zip_path)
print("Size (bytes):", zip_path.stat().st_size)

Downloaded:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\gnome\en-vi.txt.zip
Size (bytes): 5247


In [6]:
import io

with zipfile.ZipFile(
    io.BytesIO(zip_path.read_bytes()),
    "r"
) as z:
    members = z.namelist()

    print("Files inside archive:")
    for member in members:
        print(member)

    source_member = next(
        member
        for member in members
        if member.endswith(".en")
    )

    target_member = next(
        member
        for member in members
        if member.endswith(".vi")
    )

    with z.open(source_member) as src_file:
        en_lines = [
            line.decode("utf-8").rstrip("\n")
            for line in src_file
        ]

    with z.open(target_member) as tgt_file:
        vi_lines = [
            line.decode("utf-8").rstrip("\n")
            for line in tgt_file
        ]

if len(en_lines) != len(vi_lines):
    raise ValueError(
        "English và Vietnamese không có cùng số dòng."
    )

df = pd.DataFrame(
    {
        "en": en_lines,
        "vi": vi_lines
    }
)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Files inside archive:
README
LICENSE
GNOME.en-vi.en
GNOME.en-vi.vi
GNOME.en-vi.xml
Shape: (149, 2)

Columns:
['en', 'vi']

Data types:
en    str
vi    str
dtype: object


In [7]:
print("First 5 rows:")
display(df.head())

print("\nRandom 5 rows:")
display(df.sample(5, random_state=42))

First 5 rows:


,en,vi
0,Give your application an accessibility workout,Thử ra khả năng truy cập của ứng dụng
1,Accerciser Accessibility Explorer,Bộ Thám hiểm Khả năng Truy cập Accerciser
2,The default plugin layout for the bottom panel,Bố trí bổ sung mặc định cho Bảng bên dưới
3,The default plugin layout for the top panel,Bố trí bổ sung mặc định cho Bảng bên trên
4,A list of plugins that are disabled by default,Danh sách các bổ sung bị tắt theo mặc định



Random 5 rows:


,en,vi
73,Rows,Hàng
18,Interactive console for manipulating currently...,Bàn giao tiếp tương tác để thao tác điều truy ...
117,Idle,Nghỉ
78,Row,Hàng
76,Header:,Đầu trang:


In [8]:
raw_count = len(df)

missing_values = df.isna().sum()

duplicate_count = df.duplicated().sum()

print("Raw count:", raw_count)

print("\nMissing values:")
display(missing_values)

print("\nDuplicate rows:")
print(duplicate_count)

print("\nColumn statistics:")

for column in df.columns:
    print(f"\n{column}")
    print("Non-null:", df[column].notna().sum())
    print("Unique:", df[column].nunique(dropna=False))

Raw count: 149

Missing values:


en    0
vi    0
dtype: int64


Duplicate rows:
1

Column statistics:

en
Non-null: 149
Unique: 148

vi
Non-null: 149
Unique: 144


In [9]:
df_tech_candidate = df.copy()

technology_candidate_count = len(
    df_tech_candidate
)

non_technology_count = (
    raw_count - technology_candidate_count
)

print("Raw rows:", raw_count)

print(
    "Technology candidate rows:",
    technology_candidate_count
)

print(
    "Non-technology rows:",
    non_technology_count
)

Raw rows: 149
Technology candidate rows: 149
Non-technology rows: 0


In [10]:
audit_summary = {
    "source": SOURCE_NAME,
    "raw_count": int(raw_count),
    "candidate_count": int(technology_candidate_count),
    "non_technology_count": int(non_technology_count),
    "missing_values": {
        str(key): int(value)
        for key, value in missing_values.items()
    },
    "duplicate_count": int(duplicate_count),
    "columns": [
        str(column)
        for column in df.columns
    ],
    "language_pair": LANGUAGE_PAIR,
    "candidate_rule": (
        "All rows from the GNOME English-Vietnamese "
        "source are retained as technology candidates "
        "because the corpus consists of GNOME localization "
        "content. This does not establish final usable IT status."
    )
}

audit_summary

{'source': 'GNOME',
 'raw_count': 149,
 'candidate_count': 149,
 'non_technology_count': 0,
 'missing_values': {'en': 0, 'vi': 0},
 'duplicate_count': 1,
 'columns': ['en', 'vi'],
 'language_pair': 'en-vi',
 'candidate_rule': 'All rows from the GNOME English-Vietnamese source are retained as technology candidates because the corpus consists of GNOME localization content. This does not establish final usable IT status.'}

In [11]:
raw_jsonl_path = (
    RAW_DIR
    / f"{SOURCE_SHORT_NAME}_raw.jsonl"
)
raw_parquet_path = RAW_DIR / f"{SOURCE_SHORT_NAME}_raw.parquet"
audit_path = RAW_DIR / "audit_summary.json"
metadata_path = RAW_DIR / "metadata.json"

# Phase 01 tạo snapshot RAW một lần; không ghi đè artifact đã có.
phase_01_output_paths = [raw_jsonl_path, raw_parquet_path, audit_path, metadata_path]
existing_outputs = [path for path in phase_01_output_paths if path.exists()]
missing_outputs = [path for path in phase_01_output_paths if not path.exists()]
if existing_outputs and missing_outputs:
    raise RuntimeError(
        "Phát hiện RAW snapshot chưa đầy đủ; không được ghi đè hay tiếp tục. \n"
        f"Existing: {[str(path) for path in existing_outputs]}\n"
        f"Missing: {[str(path) for path in missing_outputs]}"
    )

write_raw_snapshot = not existing_outputs
if write_raw_snapshot:
    print("No existing RAW snapshot found; creating a new immutable snapshot.")
else:
    print("Complete RAW snapshot already exists; preserving it and skipping writes.")

if write_raw_snapshot:
    df.to_json(
        raw_jsonl_path,
        orient="records",
        lines=True,
        force_ascii=False
    )

print("Saved:" if write_raw_snapshot else "Preserved existing:")
print(raw_jsonl_path)

Complete RAW snapshot already exists; preserving it and skipping writes.
Preserved existing:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\gnome\gnome_raw.jsonl


In [12]:
raw_parquet_path = (
    RAW_DIR
    / f"{SOURCE_SHORT_NAME}_raw.parquet"
)

if write_raw_snapshot:
    df.to_parquet(
        raw_parquet_path,
        index=False
    )

print("Saved:" if write_raw_snapshot else "Preserved existing:")
print(raw_parquet_path)

Preserved existing:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\gnome\gnome_raw.parquet


In [13]:
import json

audit_path = RAW_DIR / "audit_summary.json"

if write_raw_snapshot:
    with open(
        audit_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            audit_summary,
            f,
            ensure_ascii=False,
            indent=2
        )

print("Saved:" if write_raw_snapshot else "Preserved existing:")
print(audit_path)

Preserved existing:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\gnome\audit_summary.json


In [14]:
metadata = {
    "source": SOURCE_NAME,
    "source_url": SOURCE_URL,
    "version_revision": DATASET_VERSION,
    "collection_date": COLLECTION_DATE,
    "download_method": DOWNLOAD_METHOD,
    "language_pair": LANGUAGE_PAIR,
    "domain": DOMAIN,
    "subcategory": "GNOME localization / software UI",
    "raw_count": int(raw_count),
    "candidate_count": int(technology_candidate_count),
    "usable_count": None,
    "notes": (
        "Raw English-Vietnamese GNOME localization corpus "
        "collected from OPUS GNOME v1. "
        "The corpus is technical software-localization data. "
        "All source rows are retained as technology candidates "
        "at this collection stage. "
        "Full language check, alignment check, cleaning, "
        "deduplication, quality/noise assessment and final "
        "IT usability confirmation have not yet been completed. "
        "License is recorded as None because the available "
        "dataset metadata currently identifies the license "
)
}

metadata_path = RAW_DIR / "metadata.json"

if write_raw_snapshot:
    with open(
        metadata_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            metadata,
            f,
            ensure_ascii=False,
            indent=2
        )

print("Saved:" if write_raw_snapshot else "Preserved existing:")
print(metadata_path)

Preserved existing:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\gnome\metadata.json


In [15]:
expected_files = [
    zip_path,
    raw_jsonl_path,
    raw_parquet_path,
    audit_path,
    metadata_path
]

verification_results = {}

for file_path in expected_files:
    verification_results[file_path.name] = file_path.is_file()

print("Final verification:\n")

for file_name, exists in verification_results.items():
    print(
        f"{file_name:45}"
        f"{'OK' if exists else 'MISSING'}"
    )

verification_passed = all(
    verification_results.values()
)

print("\nVerification passed:", verification_passed)

Final verification:

en-vi.txt.zip                                OK
gnome_raw.jsonl                              OK
gnome_raw.parquet                            OK
audit_summary.json                           OK
metadata.json                                OK

Verification passed: True


# Data Collection Status

Source:

**GNOME — OPUS-GNOME v1**

| Metric | Value |
|---|---:|
| Raw rows | Recorded after download |
| Technology candidate rows | Same as raw count |
| Non-technology rows | 0 |
| Usable IT rows | TBD |

## Completed in this notebook

- [x] Environment checked
- [x] Project root identified
- [x] Source configured
- [x] GNOME English-Vietnamese corpus downloaded
- [x] Raw parallel files converted to DataFrame
- [x] Raw schema inspected
- [x] Raw statistics recorded
- [x] Technology candidate identified
- [x] Raw JSONL saved
- [x] Raw Parquet saved
- [x] Audit summary saved
- [x] Metadata saved
- [x] Output files verified

## Not completed in this notebook

- [ ] Full language check
- [ ] Alignment check
- [ ] Data cleaning
- [ ] Deduplication
- [ ] Quality/noise assessment
- [ ] IT subdomain classification
- [ ] Final usable IT count
- [ ] Train / validation / test split

## Interpretation

GNOME is a software-localization corpus and is therefore retained
as a technology candidate at the source-collection stage.

However:

`technology_candidate_count != usable_count`

The final usable IT corpus must only be determined after the
subsequent audit, cleaning and IT-filtering stages.


> Raw data under `data/raw/` must remain unchanged.
>
> This notebook does not produce the final IT corpus.